# Digitaler Zwilling Pipeline - Automatisierte HTC-Modellierung
Dieses Notebook enthält meine vollständige Daten-Pipeline zur Erstellung eines analytischen Digitalen Zwillings. Ziel ist es, aus Simulationsdaten präzise Modelle für den Wärmeübergangskoeffizienten (HTC) zu trainieren.

Das Skript ist so aufgebaut, dass es auch auf neue thermische Simulationen angewendet werden kann. Es durchläuft dabei die folgenden fünf Schritte:

1. **ETL-Pipeline:** Einlesen der rohen Ansys-Daten im `.csv`-Format und Konvertierung in speichereffiziente Parquet-Dateien.
2. **Outlier Filtering:** Bereinigung der Daten von numerischen Gittersingularitäten (Mesh-Fehler an den Linsenkanten) mittels eines 1%-Quantilsfilters.
3. **KI-Training (Symbolic Regression):** Nutzung von `PySR`, um analytische Gleichungen für den HTC zu generieren.
4. **Validierung:** Automatisierter Test der generierten Formeln anhand des gesamten Datensatzes zur Ermittlung des tatsächlichen Fehlers (MAE).
5. **Repair:** Automatisierte Formel verbesserung

Am ende des Notebooks befindet sich die **Zentrale steuerung** mit welcher man alle funktionen Abrufen kann, dort sieht man auch immer die einzelnen outputs/Fortschritte der Codes.

### Notwendige pip installertionen

In [1]:


import sys
import subprocess

print("Prüfe und installiere benötigte Bibliotheken (Dies kann kurz dauern)...")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pandas", 
    "numpy", 
    "sympy", 
    "scikit-learn", 
    "tqdm", 
    "pysr", 
    "pyarrow",      
    "fastparquet",  
    "ipywidgets"    
])

print("Alle Bibliotheken sind einsatzbereit!")

Prüfe und installiere benötigte Bibliotheken (Dies kann kurz dauern)...
Alle Bibliotheken sind einsatzbereit!


In [2]:
import gc
import pandas as pd
import numpy as np
import re
import sympy
from tqdm import tqdm
from pathlib import Path
import tkinter as tk
from tkinter import filedialog
from pysr import PySRRegressor
from dashboard import zeige_steuerung
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


## Schritt 1: ETL-Pipeline (CSV -> Parquet)
Im ersten Schritt lese ich alle vorhandenen `.csv`-Dateien gebündelt ein. Dabei ordne ich die einzelnen Simulationsfälle (Cases) direkt ihren physikalischen Gruppen zu (beispielsweise Heiz- oder Abkühlphasen / Heatup vs. Cooldown). Um das Skript für neue Projekte wiederzuverwenden, muss lediglich das Dictionary `gruppen_zuordnung` sowie die Zuweisungslogik an die neuen Cases angepasst werden.

In [3]:
def erstelle_parquet():
    
    if Path("linsen_daten_roh.parquet").exists():
        print("Übersprungen: 'linsen_daten_roh.parquet' existiert bereits. Keine Ordner-Auswahl nötig.")
        return
    
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    
    print("Bitte wähle den Hauptordner mit den Cases aus...")
    main_folder_path = filedialog.askdirectory(title="Wähle den Ordner 'Cases'")
    if not main_folder_path:
        print("Abbruch: Kein Ordner ausgewählt.")
        return
        
    main_folder = Path(main_folder_path)
    all_tables = []
    print(f"Lese CSVs aus '{main_folder.name}' ein (Dies kann kurz dauern)...")

    # 1. ALLE DATEN ROH EINLESEN (Noch ohne Gruppen)
    regex_pattern = re.compile(r"area_Element([^_]+)_([^_]+)_time_(.*?)\.csv")
    for file_pfad in main_folder.rglob("*.csv"):
        df_temp = pd.read_csv(file_pfad)
        case_folder = next(p for p in file_pfad.parents if p.name.startswith("Case")).name
        
        hit = regex_pattern.search(file_pfad.name)
        if hit:
            linse = hit.group(1)
            surface_num = hit.group(2)
            time_val = float(hit.group(3))
            
            mapping = {'0':'bottom','1':'bottom','2':'top','3':'top','4':'lateral','5':'support'} if linse in ['1', '3'] else {'0':'bottom','1':'bottom','2':'top','3':'lateral','4':'support'}
                
            df_temp['case_id'] = case_folder
            df_temp['Linsen_art'] = linse
            df_temp['Surface'] = mapping.get(surface_num, 'unknown')
            df_temp['Time'] = time_val
            
            # HTC berechnen
            df_temp = df_temp[(df_temp['T'] != 0) & (df_temp['contact area'] != 0)]
            df_temp['HTC'] = df_temp['Q'] / (df_temp['contact area'] * df_temp['T'])
            df_temp = df_temp.drop(columns=['Q'])
            
            all_tables.append(df_temp)

    if not all_tables:
        print("Keine gültigen CSV-Dateien gefunden.")
        return

    df_final = pd.concat(all_tables, ignore_index=True)
    del all_tables
    gc.collect()

    # Speicheroptimierung: Downcast float64 zu float32
    for col in df_final.select_dtypes(include=['float64']).columns:
        df_final[col] = df_final[col].astype('float32')


    
    # 2. Clustering
    
    print("Starte KI-gestützte automatische Gruppierung...")
    
    # Für die Phase schauen wir uns die echte TEMPERATUR (T) an!
    df_trend_T = df_final.groupby(['case_id', 'Time'])['T'].mean().unstack(level=0)
    
    # Für die Gruppen (Similarity) schauen wir uns den HTC an
    df_trend_HTC = df_final.groupby(['case_id', 'Time'])['HTC'].mean().unstack(level=0)
    
    gruppen_zuordnung = {}
    phasen_zuordnung = {}
    gruppen_zaehler = {'Heatup': 1, 'Cooldown': 1}
    
    # Strengere Toleranz, da wir Durchschnittskurven vergleichen
    toleranz_mae = 10

    # A) PHASEN-ERKENNUNG (mit Temperatur)
    for case in df_trend_T.columns:
        # Temperatur ganz am Anfang vs. ganz am Ende
        t_start = df_trend_T[case].dropna().iloc[0]
        t_ende = df_trend_T[case].dropna().iloc[-1]
        
        # Wenn es am Ende wärmer ist -> Heatup!
        phasen_zuordnung[case] = 'Heatup' if t_ende > t_start else 'Cooldown'

    # B) Ähnliche Cases in Gruppen zusammenfassen
    for phase in ['Heatup', 'Cooldown']:
        cases_in_phase = [c for c, p in phasen_zuordnung.items() if p == phase]
        zugewiesen = []
        
        for case in cases_in_phase:
            if case in zugewiesen:
                continue
                
            aktuelle_gruppe = f"{phase}_Gruppe_{gruppen_zaehler[phase]}"
            gruppen_zuordnung[case] = aktuelle_gruppe
            zugewiesen.append(case)
            
            # Vergleiche diesen Case mit allen anderen
            for anderer_case in cases_in_phase:
                if anderer_case not in zugewiesen:
                    # MAE der HTC-Kurven berechnen
                    differenz = (df_trend_HTC[case] - df_trend_HTC[anderer_case]).abs().mean()
                    
                    if differenz <= toleranz_mae:
                        gruppen_zuordnung[anderer_case] = aktuelle_gruppe
                        zugewiesen.append(anderer_case)
                        
            gruppen_zaehler[phase] += 1

    print("\nAutomatische Zuweisung erfolgreich:")
    for c, g in sorted(gruppen_zuordnung.items()):
        print(f"   -> {c}: {g} (Erkannt als {phasen_zuordnung[c]})")


    # 3. ZUWEISUNG ANWENDEN UND SPEICHERN
    
    df_final['Physik_Gruppe'] = df_final['case_id'].map(gruppen_zuordnung)
    df_final['Phase'] = df_final['case_id'].map(lambda x: 1 if phasen_zuordnung[x] == 'Heatup' else 0)

    # Parquet direkt im aktuellen Ordner (wo das Notebook liegt) ablegen
    speicher_pfad = Path("./linsen_daten_roh.parquet")
    df_final.to_parquet(speicher_pfad, index=False)
    
    # .absolute() zeigt dir in der Konsole den genauen, vollen Pfad an
    print(f"\nFertig! Daten gespeichert unter:\n{speicher_pfad.absolute()}")
    del df_final, df_trend_T, df_trend_HTC
    gc.collect()

## Schritt 2: Numerisches Rauschen filtern (Mesh Outlier Removal)
In Strömungssimulationen treten an scharfen Geometriekanten (wie Lateral oder Top) häufig Gitterfehler auf, die zu physikalisch unrealistischen HTC-Spitzenwerten führen. Um die Qualität des Modelltrainings zu gewährleisten, wende ich hier einen Filter an. Für jede Fläche und jeden Case entferne ich gezielt die oberen und unteren 1 % der Datenpunkte (Extremwerte), um dieses numerische Rauschen aus den Simulationen zu eliminieren.

In [4]:
def filter_outliers():
    
    if Path("linsen_daten_roh.parquet").exists() and not Path("linsen_daten_clean.parquet").exists():
        print("Lade rohe Daten für Outlier-Filterung...")
        df = pd.read_parquet("linsen_daten_roh.parquet")
        original_zeilen = len(df)
        
        untere_grenze = df.groupby(['case_id', 'Linsen_art', 'Surface'])['HTC'].transform(lambda x: x.quantile(0.01))
        obere_grenze = df.groupby(['case_id', 'Linsen_art', 'Surface'])['HTC'].transform(lambda x: x.quantile(0.99))
        
        maske_sauber = (df['HTC'] >= untere_grenze) & (df['HTC'] <= obere_grenze)
        df_clean = df[maske_sauber].copy()
        
        entfernt = original_zeilen - len(df_clean)
        print(f"Filterung abgeschlossen! Es wurden {entfernt} kaputte Knotenpunkte ({(entfernt/original_zeilen)*100:.2f}%) entfernt.")
        
        ziel_pfad = Path("./linsen_daten_clean.parquet")
        df_clean.to_parquet(ziel_pfad, index=False)
        print(f"Saubere Daten als '{ziel_pfad.absolute()}' gespeichert.")
        
        del df, df_clean
        gc.collect()
    elif Path("linsen_daten_clean.parquet").exists():
        print("Übersprungen: 'linsen_daten_clean.parquet' existiert bereits.")
    else:
        print("Fehler: 'linsen_daten_roh.parquet' nicht gefunden. Bitte zuerst Schritt 1 ausführen.")


## Schritt 3: KI-Training (Symbolic Regression mit PySR)
In diesem Abschnitt nutze ich Symbolische Regression, um die thermodynamischen Kurven der Simulation mathematisch abzubilden. Der Algorithmus (`PySR`) kombiniert dabei vordefinierte Basisfunktionen (`+`, `-`, `*`, `/`, `exp`, `sin`, `abs`, `log`).

Um Overfitting zu vermeiden und die Robustheit der Gleichungen zu sichern, definiere ich im Code spezifische physikalische Constraints (beispielsweise das Verbot von verschachtelten Exponentialfunktionen). Das Training erfolgt separat für die zuvor definierten Modellgruppen (z.B. `Heatup_Gruppe_1`), um case-spezifische physikalische Zusammenhänge optimal abzubilden.

In [5]:
def trainiere_modelle(ziel_gruppe=None, max_iterations=250):

    
    print("Lade bereinigte Daten...")
    df = pd.read_parquet("linsen_daten_clean.parquet")
    
    df['t_skaliert'] = df['Time'] / 18000.0
    df['r'] = np.sqrt(df['x']**2 + df['y']**2)
    features_pysr = ['t_skaliert', 'x', 'y', 'z', 'r']
    
    alle_gruppen = df['Physik_Gruppe'].unique()
    if ziel_gruppe:
        alle_gruppen = [ziel_gruppe]
        
    linsen_liste = ['1', '2', '3']
    flaechen_liste = ['top', 'bottom', 'lateral', 'support']
    
    formeln_daten = []
    
    for gruppe in alle_gruppen:
        print(f"\nStarte Training für Gruppe: {gruppe}")
        alle_aufgaben = [(l, f) for l in linsen_liste for f in flaechen_liste]
        
        for linse, flaeche in tqdm(alle_aufgaben, desc=f"Trainiere {gruppe}", unit="Fläche"):
            maske = (df['Surface'] == flaeche) & (df['Linsen_art'] == linse) & (df['Physik_Gruppe'] == gruppe)
            
            # Einheitliches Downsampling für alle Flächen
            df_gefiltert = df[maske].iloc[::5]
            
            if df_gefiltert.empty:
                continue
            
            phase_name = "Heatup" if df_gefiltert['Phase'].iloc[0] == 1 else "Cooldown"
            X = df_gefiltert[features_pysr]
            y = df_gefiltert['HTC']
            
            modell = PySRRegressor(
                procs=8,
                niterations=max_iterations,
                binary_operators=["+", "-", "*", "/"],
                unary_operators=["exp", "sin", "abs", "log"],
                nested_constraints={"exp": {"exp": 0}, "sin": {"sin": 0}, "log": {"log": 0}},
                maxsize=30,
                warmup_maxsize_by=0.1,
                constraints={'/': (-1, 5), 'exp': 3},
                model_selection="best",
                verbosity=0,
                batching=True,
                batch_size=1000,
                progress=False,
                temp_equation_file=True
            )
            
            modell.fit(X, y)
            vorhersage = modell.predict(X)
            mae = mean_absolute_error(y, vorhersage)
            
            formeln_daten.append({
                'Gruppe': gruppe,
                'Phase': phase_name,
                'Linse': linse,
                'Surface': flaeche,
                'Trainings_MAE': round(mae, 3),
                'Formel': str(modell.sympy())
            })
            
    df_registry = pd.DataFrame(formeln_daten)
    
    if Path("Modell_Registry.csv").exists() and ziel_gruppe:
        df_alt = pd.read_csv("Modell_Registry.csv", sep=";")
        df_registry = pd.concat([df_alt[df_alt['Gruppe'] != ziel_gruppe], df_registry])
        
    df_registry.to_csv("Modell_Registry.csv", index=False, sep=";")
    print("\nModelle in 'Modell_Registry.csv' gespeichert.")

    del df
    gc.collect()
    print("RAM freigegeben.")

## Schritt 4: Modellauswertung & Validierung
Da für das Training aus Performancegründen nur ein ausgedünnter Datensatz (ca. 20 % der Daten) verwendet wird, erfolgt in diesem Schritt die finale Validierung. Der Code evaluiert die generierten Formeln ungeschönt anhand von 100 % der bereinigten Datenpunkte aus der `linsen_daten_clean.parquet`. Hierbei berechne ich den tatsächlichen mittleren absoluten Fehler (Echter MAE), um sicherzustellen, dass das Modell generalisiert und die Physik verstanden hat.

In [6]:
def evaluiere_zwilling():
    import gc
    import pandas as pd
    import numpy as np
    import sympy
    from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
    from pathlib import Path
    
    print("Lade bereinigte Parquet-Daten...")
    df = pd.read_parquet("linsen_daten_clean.parquet")
    df['t_skaliert'] = df['Time'] / 18000.0
    df['r'] = np.sqrt(df['x']**2 + df['y']**2)
    
    if not Path("Modell_Registry.csv").exists():
        print("Keine Registry gefunden. Bitte trainiere zuerst das Modell.")
        return
        
    df_registry = pd.read_csv("Modell_Registry.csv", sep=";")
    
    print("Starte Evaluierung (100% der Daten)...")
    
    echte_maes = []
    echte_mapes = []
    abweichungen = []
    
    symbole = {'t_skaliert': sympy.Symbol('t_skaliert'), 'x': sympy.Symbol('x'), 
               'y': sympy.Symbol('y'), 'z': sympy.Symbol('z'), 'r': sympy.Symbol('r')}
               
    for idx, row in df_registry.iterrows():
        maske = (df['Surface'] == row['Surface']) & (df['Linsen_art'] == str(row['Linse'])) & (df['Physik_Gruppe'] == row['Gruppe'])
        df_test = df[maske]
        
        if df_test.empty:
            echte_maes.append(np.nan)
            echte_mapes.append(np.nan)
            abweichungen.append(np.nan)
            continue
            
        formel_expr = sympy.sympify(row['Formel'], locals=symbole)
        benoetigte_symbole = [s.name for s in formel_expr.free_symbols]
        funktion = sympy.lambdify([symbole[s] for s in benoetigte_symbole], formel_expr, modules=['numpy'])
        
        inputs = [df_test[sym].values for sym in benoetigte_symbole]
        try:
            vorhersage = funktion(*inputs)
            if isinstance(vorhersage, (int, float)):
                vorhersage = np.full(len(df_test), vorhersage)
                
            echter_mae = mean_absolute_error(df_test['HTC'], vorhersage)
            
            relevante_daten = df_test['HTC'].abs() > 10.0
            if relevante_daten.sum() > 0:
                echter_mape = mean_absolute_percentage_error(df_test.loc[relevante_daten, 'HTC'], vorhersage[relevante_daten]) * 100
            else:
                echter_mape = np.nan
                
            abweichung = abs(row['Trainings_MAE'] - echter_mae)
            
        except Exception as e:
            print(f"Fehler bei {row['Gruppe']} {row['Linse']} {row['Surface']}: {e}")
            echter_mae, echter_mape, abweichung = np.nan, np.nan, np.nan
            
        echte_maes.append(round(echter_mae, 3))
        echte_mapes.append(round(echter_mape, 2))
        abweichungen.append(round(abweichung, 3))
        
    df_registry['Echter_MAE'] = echte_maes
    df_registry['MAPE_%'] = echte_mapes
    df_registry['Abweichung'] = abweichungen
    
    df_registry.to_csv("Modell_Registry.csv", index=False, sep=";")
    
    df_testbericht = df_registry[['Gruppe', 'Phase', 'Linse', 'Surface', 'Trainings_MAE', 'Echter_MAE', 'Abweichung', 'MAPE_%']].sort_values(by='Echter_MAE', ascending=False)
    df_testbericht.to_csv("Master_Testbericht.csv", index=False, sep=";", decimal=",")
    
    print("\nEvaluierung abgeschlossen.")
    print("Modell_Registry.csv wurde geupdatet.")
    print("Master_Testbericht.csv wurde generiert.")
    
    del df
    gc.collect()

## Schritt 5: Auto-Repair
Dieses Skript liest die Evaluierungsergebnisse aus der Registry. Modelle, die einen festgelegten Schwellenwert (z.B. MAE > 60) überschreiten, werden automatisch identifiziert. Das Skript passt die Trainingsstrategie für diese spezifischen Problemfälle an (z.B. Nutzung von 100% der Daten und längere Formeln) und trainiert sie neu. Bei einer Verbesserung wird die Registry überschrieben.

In [7]:
def auto_repair_modelle(schwellenwert_mae=60.0):
    print("Lade bereinigte Daten für Reparatur...")
    df = pd.read_parquet("linsen_daten_clean.parquet")
    df['t_skaliert'] = df['Time'] / 18000.0
    df['r'] = np.sqrt(df['x']**2 + df['y']**2)
    features_pysr = ['t_skaliert', 'x', 'y', 'z', 'r']

    if not Path("Modell_Registry.csv").exists():
        print("Keine Registry gefunden. Bitte zuerst trainieren und evaluieren.")
        return

    df_registry = pd.read_csv("Modell_Registry.csv", sep=";")

    if 'Echter_MAE' not in df_registry.columns:
        print("Echter_MAE nicht in Registry. Bitte zuerst Evaluierung (Schritt 4) ausführen.")
        return

    problemfaelle = df_registry[df_registry['Echter_MAE'] > schwellenwert_mae]

    if problemfaelle.empty:
        print(f"Alle Modelle liegen unter dem Schwellenwert von {schwellenwert_mae} W/m²K. Keine Reparatur nötig.")
        del df
        gc.collect()
        return

    print(f"Starte Auto-Repair für {len(problemfaelle)} Modelle...")

    for idx, row in problemfaelle.iterrows():
        gruppe = row['Gruppe']
        linse = str(row['Linse'])
        flaeche = row['Surface']
        alter_mae = row['Echter_MAE']

        print(f"\nRepariere: {gruppe} | Linse {linse} | {flaeche} (Alter MAE: {alter_mae:.2f})")

        maske = (df['Surface'] == flaeche) & (df['Linsen_art'] == linse) & (df['Physik_Gruppe'] == gruppe)

        if flaeche in ['support', 'lateral']:
            df_gefiltert = df[maske]
            maxsize_val = 40
            nested = {"exp": {"exp": 1}, "sin": {"sin": 1}, "log": {"log": 1}}
        else:
            df_gefiltert = df[maske].iloc[::2]
            maxsize_val = 35
            nested = {"exp": {"exp": 0}, "sin": {"sin": 0}, "log": {"log": 0}}

        if df_gefiltert.empty:
            continue

        X = df_gefiltert[features_pysr]
        y = df_gefiltert['HTC']

        modell = PySRRegressor(
            procs=8,
            niterations=500,
            binary_operators=["+", "-", "*", "/"],
            unary_operators=["exp", "sin", "abs", "log"],
            nested_constraints=nested,
            maxsize=maxsize_val,
            warmup_maxsize_by=0.1,
            constraints={'/': (-1, 5), 'exp': 3},
            model_selection="best",
            verbosity=0,
            batching=True,
            batch_size=1000,
            progress=False,
            temp_equation_file=True
        )

        modell.fit(X, y)
        vorhersage = modell.predict(X)
        neuer_trainings_mae = mean_absolute_error(y, vorhersage)

        X_voll = df[maske][features_pysr]
        y_voll = df[maske]['HTC']
        vorhersage_voll = modell.predict(X_voll)
        neuer_echter_mae = mean_absolute_error(y_voll, vorhersage_voll)

        if neuer_echter_mae < alter_mae:
            print(f"Verbesserung erzielt. Neuer MAE: {neuer_echter_mae:.2f} W/m²K")
            df_registry.loc[idx, 'Formel'] = str(modell.sympy())
            df_registry.loc[idx, 'Trainings_MAE'] = round(neuer_trainings_mae, 3)
            df_registry.loc[idx, 'Echter_MAE'] = round(neuer_echter_mae, 3)
            df_registry.loc[idx, 'Abweichung'] = round(abs(neuer_trainings_mae - neuer_echter_mae), 3)
        else:
            print(f"Keine Verbesserung (Neuer MAE: {neuer_echter_mae:.2f} W/m²K). Altes Modell wird behalten.")

    df_registry.to_csv("Modell_Registry.csv", index=False, sep=";")
    print("\nAuto-Repair abgeschlossen. Registry wurde aktualisiert.")

    del df
    gc.collect()

## Zentrale Steuerung

In [9]:
def starte_training(ziel_gruppe=None, max_iterations=1000):
    trainiere_modelle(ziel_gruppe=ziel_gruppe, max_iterations=max_iterations)

def starte_repair():
    auto_repair_modelle(schwellenwert_mae=60.0)

# Rufe das UI auf und verknüpfe es
zeige_steuerung(
    f_etl=erstelle_parquet,
    f_filter=filter_outliers,
    f_train=starte_training,  
    f_eval=evaluiere_zwilling,
    f_repair=starte_repair
)

Output(layout=Layout(border_bottom='1px solid #bdc3c7', border_left='1px solid #bdc3c7', border_right='1px sol…